# Cruzamento das Bases — Observatório do Clima

**Squad Observatório do Clima – Grande Desafio DataConnect Solutions**

Este notebook faz o JOIN final das três bases já tratadas:

1. `sinisa_limpo.csv` — base do módulo Águas Pluviais do SINISA (5.570 municípios, tabela mestra).
2. `CAPAG-2025_Tratamento.csv` — capacidade de pagamento (nota CAPAG) por município.
3. `S2iD_Desastres_Hidrologicos_Selecao_Auditada.csv` — histórico de desastres hidrológicos do Atlas (1 linha por desastre).

Saída: `dataset_mestre_observatorio.csv`, 1 linha por município, com os 5.570
municípios originais do SINISA preservados (left join).

## 0. Configuração do ambiente e upload dos arquivos

In [1]:
import glob
import pandas as pd

pd.set_option("display.max_columns", 50)

try:
    from google.colab import files
    print("Ambiente Colab detectado. Selecione os 3 CSVs abaixo:")
    uploaded = files.upload()
except ImportError:
    print("Ambiente não é o Colab: assumindo que os arquivos já estão no diretório atual.")

def encontrar_arquivo(padrao):
    candidatos = glob.glob(padrao)
    if not candidatos:
        raise FileNotFoundError(f"Não encontrei nenhum arquivo correspondente a '{padrao}'.")
    if len(candidatos) > 1:
        print(f"Aviso: mais de um arquivo bate com '{padrao}': {candidatos}. Usando o primeiro.")
    return candidatos[0]

CAMINHO_SINISA = encontrar_arquivo("sinisa_limpo.csv")
CAMINHO_CAPAG = encontrar_arquivo("capag_nacional_tratada.csv")  # CAPAG set/2026 (Laura, 08/10)
CAMINHO_S2ID = encontrar_arquivo("*S2iD*.csv")

print("SINISA:", CAMINHO_SINISA)
print("CAPAG:", CAMINHO_CAPAG)
print("S2iD:", CAMINHO_S2ID)

Ambiente não é o Colab: assumindo que os arquivos já estão no diretório atual.
SINISA: sinisa_limpo.csv
CAPAG: capag_nacional_tratada.csv
S2iD: S2iD_Desastres_Hidrologicos_Selecao_Auditada.csv


## 1. Leitura blindada

Forçamos as colunas de código IBGE a `str` direto no `read_csv` — evita a
"maldição do decimal" (`3302502` virar `3302502.0`) e qualquer perda de zero
à esquerda.

Dois detalhes que os arquivos brutos exigem:
- Os três arquivos têm **BOM UTF-8** (`\ufeff`) no início — por isso usamos
  `encoding='utf-8-sig'`, senão o nome da primeira coluna de cada arquivo sai
  como `'\ufeffCod_IBGE_7'` em vez de `'Cod_IBGE_7'` e qualquer referência
  direta à coluna quebra silenciosamente.
- O S2iD usa `;` como separador, os outros dois usam `,`.

In [2]:
df_sinisa = pd.read_csv(
    CAMINHO_SINISA,
    dtype={"Cod_IBGE_7": str},
    encoding="utf-8-sig",
)

df_capag = pd.read_csv(
    CAMINHO_CAPAG,
    dtype={"cod_ibge": str},
    encoding="utf-8-sig",
)

df_s2id = pd.read_csv(
    CAMINHO_S2ID,
    sep=";",
    dtype={"Cod_IBGE_Mun": str},
    encoding="utf-8-sig",
)

print("df_sinisa:", df_sinisa.shape, "| colunas:", list(df_sinisa.columns))
print("df_capag: ", df_capag.shape, "| colunas:", list(df_capag.columns))
print("df_s2id:  ", df_s2id.shape, "| colunas:", list(df_s2id.columns))

# Trava de versão: a CAPAG tem que ser a de setembro/2026
contagem_capag = df_capag["capag"].value_counts()
print("\nNotas CAPAG:", contagem_capag.to_dict())
assert contagem_capag.get("C", 0) == 2274 and contagem_capag.get("D", 0) == 20, "Esta CAPAG não é a de setembro/2026!"

df_sinisa: (5570, 9) | colunas: ['Cod_IBGE_7', 'Nome', 'UF', 'mapeamento_area_risco', 'qtd_domicilios_risco', 'municipio_critico', 'pct_vias_rede_drenagem', 'investimento_percapita_drenagem', 'pct_domicilios_risco']
df_capag:  (5567, 9) | colunas: ['cod_ibge', 'municipio', 'uf', 'capag_pre_ranking', 'ranking_cconf', 'capag', 'criterio_capag_cd', 'sem_nota_capag', 'criterio_fiscal_priorizacao']
df_s2id:   (9890, 17) | colunas: ['Cod_IBGE_Mun', 'Nome_Municipio', 'Sigla_UF', 'Data_Evento', 'Status', 'descricao_tipologia', 'Protocolo_S2iD', 'DM_total_danos_materiais', 'PEPL_total_publico', 'DH_total_calculado', 'flag_sem_declaracao_financeira', 'flag_inconsistencia_soma_humana', 'flag_reconhecido_sem_impacto', 'flag_duplicata_protocolo', 'flag_duplicata_semantica', 'flag_outlier_danos_materiais', 'flag_outlier_humanos']

Notas CAPAG: {'C': 2274, 'A': 1096, 'B': 963, 'n.d.': 426, 'A+': 418, 'B+': 200, 'n.e.': 170, 'D': 20}


### 1.1 Validações rápidas das chaves

Antes de cruzar, confirmamos que as colunas de código IBGE vieram como texto,
com 7 dígitos, sem duplicatas (nas bases que devem ser 1 linha por
município) e sem nulos.

In [3]:
def checar_chave(df, coluna, nome_df, esperar_unico=True):
    print(f"--- {nome_df}.{coluna} ---")
    print(f"  dtype: {df[coluna].dtype}")
    print(f"  nulos: {df[coluna].isna().sum()}")
    tamanhos = df[coluna].dropna().str.len().value_counts()
    print(f"  tamanhos encontrados: {dict(tamanhos)}")
    duplicados = df[coluna].duplicated().sum()
    print(f"  duplicados: {duplicados}" + ("  <-- ESPERADO 0, VERIFIQUE!" if esperar_unico and duplicados else ""))
    print()

checar_chave(df_sinisa, "Cod_IBGE_7", "df_sinisa", esperar_unico=True)
checar_chave(df_capag, "cod_ibge", "df_capag", esperar_unico=True)
checar_chave(df_s2id, "Cod_IBGE_Mun", "df_s2id", esperar_unico=False)  # 1 linha por desastre, não por município

--- df_sinisa.Cod_IBGE_7 ---
  dtype: str
  nulos: 0
  tamanhos encontrados: {7: np.int64(5570)}
  duplicados: 0

--- df_capag.cod_ibge ---
  dtype: str
  nulos: 0
  tamanhos encontrados: {7: np.int64(5567)}
  duplicados: 0

--- df_s2id.Cod_IBGE_Mun ---
  dtype: str
  nulos: 0
  tamanhos encontrados: {7: np.int64(9890)}
  duplicados: 6779



## 2. Tratamento pré-join

### 2.1 SINISA — flag `participou_sinisa`

`True` quando o município respondeu ao formulário (ou seja,
`mapeamento_area_risco` não é nulo), `False` quando não participou.

In [4]:
df_sinisa["participou_sinisa"] = df_sinisa["mapeamento_area_risco"].notna()

print(df_sinisa["participou_sinisa"].value_counts())

participou_sinisa
True     4845
False     725
Name: count, dtype: int64


### 2.2 S2iD — agregando de desastre (linha) para município (linha única)

A base traz uma linha por desastre; agregamos por `Cod_IBGE_Mun` antes do
join para não multiplicar linhas do SINISA no merge.

In [5]:
df_s2id_agrupado = (
    df_s2id
    .groupby("Cod_IBGE_Mun")
    .agg(
        total_desastres=("Protocolo_S2iD", "count"),
        soma_danos_materiais=("DM_total_danos_materiais", "sum"),
        soma_prejuizo_publico=("PEPL_total_publico", "sum"),
        danos_humanos=("DH_total_calculado", "sum"),
    )
    .reset_index()
)

total_dh = int(df_s2id_agrupado["danos_humanos"].sum())
print("Municípios com pelo menos 1 desastre registrado:", df_s2id_agrupado.shape[0])
print("Total de danos humanos:", total_dh)
if total_dh != 3525510:
    print("AVISO: o esperado com os feridos é 3.525.510. Este CSV do Atlas ainda é a versão sem os feridos;"
          " a coluna danos_humanos fica subestimada até a Vitória subir o CSV corrigido.")
df_s2id_agrupado.head()

Municípios com pelo menos 1 desastre registrado: 3111
Total de danos humanos: 3525510


,Cod_IBGE_Mun,total_desastres,soma_danos_materiais,soma_prejuizo_publico,danos_humanos
0,1100015,2,592472.55,8540333.04,237
1,1100023,2,1664757.10,1669780.44,115
2,1100049,5,7935576.74,0.00,1552
3,1100056,1,0.00,0.00,6
4,1100080,1,284717.59,15029.20,0


## 3. O grande cruzamento (left join)

`df_sinisa` é a tabela da esquerda — garante que os 5.570 municípios do
SINISA permanecem intactos no resultado final, com ou sem match nas outras
duas bases.

`validate="many_to_one"` em cada merge é uma checagem defensiva: se algum dia
a CAPAG ou o S2iD agrupado tiver uma chave duplicada, o merge quebra com erro
explícito em vez de silenciosamente multiplicar linhas do SINISA.

In [6]:
linhas_antes = len(df_sinisa)

# --- SINISA + CAPAG (nota, critério C/D e as flags da decisão 1) ---
df_temp = df_sinisa.merge(
    df_capag[["cod_ibge", "capag", "criterio_capag_cd", "sem_nota_capag", "criterio_fiscal_priorizacao"]],
    left_on="Cod_IBGE_7",
    right_on="cod_ibge",
    how="left",
    validate="many_to_one",
)
print(f"Após merge com CAPAG: {len(df_temp)} linhas "
      f"({df_temp['cod_ibge'].notna().sum()} municípios encontraram match na CAPAG).")
assert len(df_temp) == linhas_antes, "O merge com a CAPAG alterou o número de linhas do SINISA!"

# --- (SINISA + CAPAG) + S2iD agrupado ---
df_final = df_temp.merge(
    df_s2id_agrupado,
    left_on="Cod_IBGE_7",
    right_on="Cod_IBGE_Mun",
    how="left",
    validate="many_to_one",
)
print(f"Após merge com S2iD agrupado: {len(df_final)} linhas "
      f"({df_final['Cod_IBGE_Mun'].notna().sum()} municípios com histórico de desastre).")
assert len(df_final) == linhas_antes, "O merge com o S2iD alterou o número de linhas do SINISA!"

Após merge com CAPAG: 5570 linhas (5567 municípios encontraram match na CAPAG).
Após merge com S2iD agrupado: 5570 linhas (3111 municípios com histórico de desastre).


### 3.1 Municípios sem linha na CAPAG

Registro de quem não encontrou par na CAPAG. Com a versão de abril eram 3: Brasília/DF (o Distrito Federal não é município e não tem CAPAG municipal), Fernando de Noronha/PE (distrito estadual) e Calçoene/AP (ausente no arquivo do Tesouro).

Atenção para a decisão 1 (filtro "quem não tem A ou B"): um filtro que só exclui A e B deixa passar quem não tem linha na CAPAG. A célula precisa decidir se "sem linha na CAPAG" conta como "sem nota".

In [7]:
sem_capag = df_final[df_final["capag"].isna()][["Cod_IBGE_7", "Nome", "UF"]]
print(f"{len(sem_capag)} municípios sem linha na CAPAG:")
print(sem_capag.to_string(index=False))

3 municípios sem linha na CAPAG:
Cod_IBGE_7                Nome UF
   1600204            Calçoene AP
   2605459 Fernando de Noronha PE
   5300108            Brasília DF


## 4. Limpeza pós-join e exportação

- Remove `cod_ibge` e `Cod_IBGE_Mun` (chaves duplicadas do merge) — só
  `Cod_IBGE_7` permanece como chave primária.
- Município sem match no S2iD = 0 desastres, não nulo — preenchemos as 3
  colunas agregadas com `0`.

In [8]:
df_final = df_final.drop(columns=["cod_ibge", "Cod_IBGE_Mun"])

colunas_s2id_agregadas = ["total_desastres", "soma_danos_materiais", "soma_prejuizo_publico", "danos_humanos"]
nulos_antes = df_final[colunas_s2id_agregadas].isna().sum()

df_final[colunas_s2id_agregadas] = df_final[colunas_s2id_agregadas].fillna(0)
df_final["total_desastres"] = df_final["total_desastres"].astype(int)

print("Nulos preenchidos com 0 em cada coluna agregada do S2iD:")
print(nulos_antes)

# Marcações do Atlas (decisão 2 usa tem_dano_atlas)
df_final["tem_registro_atlas"] = df_final["total_desastres"] > 0
df_final["tem_dano_atlas"] = (df_final["soma_danos_materiais"] + df_final["soma_prejuizo_publico"]) > 0
print("\nMunicípios com registro no Atlas:", int(df_final["tem_registro_atlas"].sum()))
print("Municípios com dano em R$ no Atlas:", int(df_final["tem_dano_atlas"].sum()))

Nulos preenchidos com 0 em cada coluna agregada do S2iD:
total_desastres          2459
soma_danos_materiais     2459
soma_prejuizo_publico    2459
danos_humanos            2459
dtype: int64

Municípios com registro no Atlas: 3111
Municípios com dano em R$ no Atlas: 2702


In [9]:
assert len(df_final) == 5570, f"Esperava 5.570 municípios, saíram {len(df_final)}."
assert df_final["Cod_IBGE_7"].duplicated().sum() == 0, "Chave primária Cod_IBGE_7 com duplicata!"
assert df_final[colunas_s2id_agregadas].isna().sum().sum() == 0, "Ainda há NaN nas colunas agregadas do S2iD."

print(f"Dataset final: {df_final.shape[0]} municípios, {df_final.shape[1]} colunas.")
print("Colunas:", list(df_final.columns))
df_final.head()

Dataset final: 5570 municípios, 20 colunas.
Colunas: ['Cod_IBGE_7', 'Nome', 'UF', 'mapeamento_area_risco', 'qtd_domicilios_risco', 'municipio_critico', 'pct_vias_rede_drenagem', 'investimento_percapita_drenagem', 'pct_domicilios_risco', 'participou_sinisa', 'capag', 'criterio_capag_cd', 'sem_nota_capag', 'criterio_fiscal_priorizacao', 'total_desastres', 'soma_danos_materiais', 'soma_prejuizo_publico', 'danos_humanos', 'tem_registro_atlas', 'tem_dano_atlas']


,Cod_IBGE_7,Nome,UF,mapeamento_area_risco,qtd_domicilios_risco,municipio_critico,pct_vias_rede_drenagem,investimento_percapita_drenagem,pct_domicilios_risco,participou_sinisa,capag,criterio_capag_cd,sem_nota_capag,criterio_fiscal_priorizacao,total_desastres,soma_danos_materiais,soma_prejuizo_publico,danos_humanos,tem_registro_atlas,tem_dano_atlas
0,1100015,Alta Floresta D Oeste,RO,Sim,150.0,Não,27.12,0.00,2.27,True,C,True,False,True,2,592472.55,8540333.04,237.0,True,True
1,1100023,Ariquemes,RO,Sim,53.0,Sim,32.28,5.08,0.13,True,B+,False,False,False,2,1664757.10,1669780.44,115.0,True,True
2,1100031,Cabixi,RO,Não,0.0,Não,NaN,0.00,0.00,True,A,False,False,False,0,0.00,0.00,0.0,False,False
3,1100049,Cacoal,RO,Sim,430.0,Sim,22.66,34.27,1.14,True,B+,False,False,False,5,7935576.74,0.00,1552.0,True,True
4,1100056,Cerejeiras,RO,Não,0.0,Não,10.29,13.95,0.00,True,A+,False,False,False,1,0.00,0.00,6.0,True,False


In [10]:
CAMINHO_SAIDA = "dataset_mestre_observatorio.csv"
df_final.to_csv(CAMINHO_SAIDA, index=False, encoding="utf-8-sig")

print(f"Exportado: {CAMINHO_SAIDA} ({len(df_final)} linhas, {df_final.shape[1]} colunas)")

try:
    from google.colab import files as colab_files
    colab_files.download(CAMINHO_SAIDA)
except ImportError:
    print("Fora do Colab: o arquivo foi salvo no diretório atual do notebook.")

Exportado: dataset_mestre_observatorio.csv (5570 linhas, 20 colunas)
Fora do Colab: o arquivo foi salvo no diretório atual do notebook.
